# APP

In [1]:
import os
from dotenv import load_dotenv
from groq import Groq
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
if os.path.exists('../.env'):
    load_dotenv()
    print("[INFO] Environment variables were loaded.")
else:
    print("[WARNING] File .env was not found. Some settings may be missing.")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("[ERROR] GROQ_API_KEY was not found in .env!")

[INFO] Environment variables were loaded.


## Utils

### Embeddings

In [3]:
import os
from langchain_huggingface import HuggingFaceEmbeddings

os.environ["TOKENIZERS_PARALLELISM"] = "false"

class Embeddings:
    def __init__(self):
        model_name = "BAAI/bge-base-en"
        encode_kwargs = {'normalize_embeddings': True} 
        self.model = HuggingFaceEmbeddings(
            model_name = model_name, 
            model_kwargs={'device': 'cpu'},
            encode_kwargs = encode_kwargs
            )

    def get_embedding_model(self):
        return self.model

c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Retriever

In [4]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS

class Retriever:
    def __init__(self, embedding):
        self.embedding = embedding.get_embedding_model()
        self.documents = []
        self.vector_store = None

    def load_documents(self, documents_path):
        for filename in os.listdir(documents_path):
            if filename.endswith(".pdf"):
                loader = PyMuPDFLoader(os.path.join(documents_path, filename))
                loaded_docs = loader.load()
                self.documents.extend(loaded_docs)

    def create_vectordb(self):
        splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
        docs_split = splitter.split_documents(self.documents)
        self.vector_store = FAISS.from_documents(docs_split, self.embedding)

    def retrieve(self, query, k = 5):
        if not self.vector_store:
            raise ValueError("Vector store is not initialized.")
        
        docs = self.vector_store.similarity_search(query, k = k)
        
        return docs

---

## Main Classes

### DocumentAgent

In [5]:
from groq import Groq

class DocumentAgent:
    def __init__(self, api_key):
        self.client = Groq(api_key = api_key)

    def summarize_documents(self, documents, query):
        context = "\n\n".join([doc.page_content for doc in documents])

        messages = [
            {"role": "system", "content": "You summarize documents with precision."},
            {"role": "user", "content": f"Documents: {context}\n\nAnswer the question briefly: {query}"}
        ]

        completion = self.client.chat.completions.create(model = "openai/gpt-oss-120b",
                                                         messages = messages,
                                                         temperature = 0.7,
                                                         max_tokens = 2048)
        
        

        return completion.choices[0].message.content.strip()

### ReasoningAgent

In [6]:
from groq import Groq

class ReasoningAgent:
    def __init__(self, api_key):
        self.client = Groq(api_key = api_key)

    def generate_reasoning(self, summary, query):
        messages = [
            {"role": "system", "content": "You are a specialist in logical reasoning about texts."},
            {"role": "user", "content": f"Based on the summary: {summary}\n\nDo a critical analyse to answer the question:: {query}"}
        ]

        completion = self.client.chat.completions.create(model = "openai/gpt-oss-120b",
                                                         messages = messages,
                                                         temperature = 0.7,
                                                         max_tokens = 2048)
        return completion.choices[0].message.content.strip()

### MetaAgent

In [7]:
from groq import Groq

class MetaAgent:
    def __init__(self, api_key):
        self.client = Groq(api_key = api_key)

    def generate_final_answer(self, summary, reasoning, query):
        messages = [
            {"role": "system", "content": "You generate clear and detailed answers consolidating the information."},
            {"role": "user", "content": f"Original question: {query}\n\nSummary: {summary}\n\nLogical Reasoning: {reasoning}\n\nProvide the consolidated and detailed answer:"}
        ]

        completion = self.client.chat.completions.create(model = "openai/gpt-oss-120b",
                                                         messages = messages,
                                                         temperature = 0.7,
                                                         max_tokens = 2048,
                                                         stream = True)

        final_answer = ""

        for chunk in completion:
            chunk_content = chunk.choices[0].delta.content or ""
            final_answer += chunk_content

        return final_answer


---

## Main block

In [8]:
# Embedding
embedding_model = Embeddings()

# Retriever RAG
retriever = Retriever(embedding_model)
retriever.load_documents("data/documents/")
retriever.create_vectordb()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14417.15it/s]


In [9]:
class AgentState(TypedDict):
    query: str
    documents: List[str]
    summary: str
    reasoning: str
    final_answer: str

In [10]:
def node_document_agent(state: AgentState) -> dict:
    agent = DocumentAgent(GROQ_API_KEY)
    documents = retriever.retrieve(state['query'])
    summary = agent.summarize_documents(documents, state['query'])
    # print("\n[Documents Summary]:", summary)
    return {'documents': documents, 'summary': summary}

In [11]:
def node_reasoning_agent(state: AgentState) -> dict:
    agent = ReasoningAgent(GROQ_API_KEY)

    reasoning = agent.generate_reasoning(state['summary'], state['query'])
    # print("\n[Reasoning]:", reasoning)
    return {'reasoning': reasoning}

In [12]:
def node_meta_agent(state: AgentState) -> dict:
    agent = MetaAgent(GROQ_API_KEY)
    final_answer = agent.generate_final_answer(state['summary'], state['reasoning'], state['query'])

    print("\n[Final Answer]:", final_answer)
    return {'final_answer': final_answer}

In [13]:
workflow = StateGraph(AgentState)

# Adding agents graph nodes
workflow.add_node("document_agent", node_document_agent)
workflow.add_node("reasoning_agent", node_reasoning_agent)
workflow.add_node("meta_agent", node_meta_agent)

# Setting the entry point of the flow
workflow.set_entry_point("document_agent")

# Setting the transition between nodes and graph
workflow.add_edge("document_agent", "reasoning_agent")
workflow.add_edge("reasoning_agent", "meta_agent")
workflow.add_edge("meta_agent", END)

# Compiling the graph into a exe application
workflow_app = workflow.compile()

In [15]:
# What is the object of the contract?
# What are the obligations of the contracting party?
# In case of termination due to breach of any obligation, what is the amount of a compensatory fine that the defaulting party will be subject to?

query = "In case of termination due to breach of any obligation, what is the amount of a compensatory fine that the defaulting party will be subject to?"
initial_state = AgentState(query = query, documents = [], summary = "", reasoning = "", final_answer = "")
workflow_app.invoke(initial_state)

print("\nProcessing complete. Thank you.!\n")


[Final Answer]: **Compensatory Fine for Termination Due to Breach**

| Item | Details |
|------|---------|
| **Amount of the fine** | **R$ 15.000,00** (fifteen thousand reais) |
| **When it applies** | Whenever one party terminates the agreement because the other party has failed to fulfil any contractual obligation. |
| **Legal basis** | • **Art. 475‑J, CC** – permits a contractual penalty (cláusula penal) for non‑performance. <br>• **Art. 413, CC** – the penalty must be “adequada” and not “excessivamente onerosa”. The parties have freely agreed on the fixed amount of R$ 15.000,00, which is presumed enforceable unless proven manifestly disproportionate. |
| **Purpose of the fine** | 1. **Deterrence** – discourages breaches. <br>2. **Compensation** – pre‑established amount to offset administrative, operational and reputational losses caused by the abrupt termination. <br>3. **Simplification** – avoids a lengthy damage‑assessment process, giving certainty to both parties. |
| **Adequac